In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, SubsetRandomSampler
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import sys
import random

root_path = r"c:\Users\HD\Documents\GitHub\rPPG"

if root_path not in sys.path:
    sys.path.insert(0, root_path)

# Assuming these are available in your environment based on provided files
from dataset.data_loader.InfantDataLoader import InfantDataLoader
from neural_methods.loss.NegPearsonLoss import Neg_Pearson
from neural_methods.model.iBVPNet import iBVPNet
# Note: Ensure the modified calculate_metrics function from our previous conversation is imported here
from evaluation.metrics import calculate_metrics 

class ConfigMock:
    def __init__(self, dictionary):
        for key, value in dictionary.items():
            if isinstance(value, dict):
                setattr(self, key, ConfigMock(value))
            else:
                setattr(self, key, value)

RANDOM_SEED = 100
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [ ]:
class CombinedLoss(nn.Module):
    """Allows using MSE, NegPearson, or a weighted combination of both."""
    def __init__(self, w_mse=1.0, w_np=1.0):
        super().__init__()
        self.w_mse = w_mse
        self.w_np = w_np
        self.mse_loss = nn.MSELoss()
        self.np_loss = Neg_Pearson()

    def forward(self, preds, labels):
        loss = 0.0
        if self.w_mse > 0:
            loss += self.w_mse * self.mse_loss(preds, labels)
        if self.w_np > 0:
            loss += self.w_np * self.np_loss(preds, labels)
        return loss

def train_and_validate(model, train_loader, val_loader, fold_idx, hyperparams, config):
    """
    Trains the model, validates it, saves best models based on Loss, 
    and returns the metrics row corresponding to the best validation loss.
    """
    device = hyperparams['device']
    optimizer = optim.Adam(model.parameters(), lr=hyperparams['lr'])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=hyperparams['lr'], 
        epochs=hyperparams['epochs'], 
        steps_per_epoch=len(train_loader)
    )
    criterion = CombinedLoss(w_mse=hyperparams['w_mse'], w_np=hyperparams['w_np'])

    best_val_loss = float('inf')
    best_val_metrics = None
    
    # Tracking for plots
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    for epoch in range(hyperparams['epochs']):
        print(f"\n--- Fold {fold_idx} | Epoch {epoch+1}/{hyperparams['epochs']} ---")
        
        # --- TRAINING ---
        model.train()
        train_loss = 0.0
        
        for batch in tqdm(train_loader, desc="Training", ncols=80):
            # Batch components: data, label, filename (subj), chunk_id
            data, labels = batch[0].to(device), batch[1].to(device)

            # iBVPNet Specific temporal padding to match output dims
            if hyperparams['model_name'] == "iBVPNet":
                last_frame = torch.unsqueeze(data[:, :, -1, :, :], 2)
                data = torch.cat((data, last_frame), 2)

            optimizer.zero_grad()
            preds = model(data)
            
            # Normalize internal features per iBVPNetTrainer pattern
            preds = (preds - torch.mean(preds)) / torch.std(preds)
            labels = (labels - torch.mean(labels)) / torch.std(labels)
            
            loss = criterion(preds, labels)
            loss.backward()
            
            history['lr'].append(scheduler.get_last_lr()[0])
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        # --- VALIDATION ---
        model.eval()
        val_loss = 0.0
        
        # Dictionary setup for metric calculation 
        predictions_dict = {}
        labels_dict = {}

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation", ncols=80):
                data, labels = batch[0].to(device), batch[1].to(device)
                subj_ids = batch[2]
                chunk_ids = batch[3]

                if hyperparams['model_name'] == "iBVPNet":
                    last_frame = torch.unsqueeze(data[:, :, -1, :, :], 2)
                    data = torch.cat((data, last_frame), 2)

                preds = model(data)
                
                preds_norm = (preds - torch.mean(preds)) / torch.std(preds)
                labels_norm = (labels - torch.mean(labels)) / torch.std(labels)
                
                loss = criterion(preds_norm, labels_norm)
                val_loss += loss.item()

                # Structure data for `calculate_metrics`
                for idx in range(data.shape[0]):
                    subj = subj_ids[idx]
                    chunk = int(chunk_ids[idx])
                    if subj not in predictions_dict:
                        predictions_dict[subj] = {}
                        labels_dict[subj] = {}
                    predictions_dict[subj][chunk] = preds[idx].cpu()
                    labels_dict[subj][chunk] = labels[idx].cpu()

        avg_val_loss = val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)
        
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # Check for lowest validation loss
        is_best_loss = avg_val_loss < best_val_loss
        if is_best_loss:
            best_val_loss = avg_val_loss
            model_save_path = os.path.join(OUTPUT_DIR, f"{hyperparams['model_name']}_Fold{fold_idx}_BestLoss.pth")
            torch.save(model.state_dict(), model_save_path)
            #print(f"[*] Best validation loss achieved ({best_val_loss:.4f}). Model saved to {model_save_path}")

            epoch_metrics = calculate_metrics(
                predictions=predictions_dict, 
                labels=labels_dict, 
                config=config, 
                filename_id=f"{hyperparams['model_name']}_Fold{fold_idx}_Epoch{epoch}",
                ba=is_best_loss, 
                save_ba=is_best_loss
            )
        
        # if is_best_loss:
            best_val_metrics = epoch_metrics
            #print(f"[+] Metrics stored for current best validation loss.")

    # --- PLOT LOSS & LR ---
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(history['train_loss'], label='Train Loss', color='blue')
    ax1.plot(history['val_loss'], label='Val Loss', color='orange')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend(loc='upper left')
    
    ax2 = ax1.twinx()
    # Average LR per epoch just for visualization scaling
    lrs_per_epoch = [np.mean(history['lr'][i*len(train_loader):(i+1)*len(train_loader)]) for i in range(hyperparams['epochs'])]
    ax2.plot(lrs_per_epoch, label='Learning Rate', color='green', linestyle='dashed', alpha=0.5)
    ax2.set_ylabel('Learning Rate')
    ax2.legend(loc='upper right')
    
    plt.title(f"{hyperparams['model_name']} - Fold {fold_idx} Metrics")
    plt.savefig(os.path.join(OUTPUT_DIR, f"{hyperparams['model_name']}_Fold{fold_idx}_TrainingPlot.png"))
    plt.close()

    return best_val_metrics


In [ ]:
MODEL_NAME = "iBVPNet"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "./kfold_outputs"

# Training Parameters
EPOCHS = 1
BATCH_SIZE = 8
LR = 1e-4

# Dataloaders Parameters
NUM_WORKERS = 0
PIN_MEMORY = True
PREFETCH_FACTOR = None
PERSISTENT_WORKERS = False

# Model Parameters
IN_CHANNELS = 3
FRAMES = 150

# Adaptive Loss Weights (Set to 0 to disable a specific loss)
WEIGHT_MSE = 0.5
WEIGHT_NEG_PEARSON = 0.5

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Minimal config required by InfantDataLoader and calculate_metrics
loader_config_dict = {
    'CACHED_PATH': 'G:/preprocessed_data/preprocessed_cache_72x_150f_d-raw_l-raw',
    'FILE_LIST_PATH': 'G:/preprocessed_data/preprocessed_cache_72x_150f_d-raw_l-raw.csv',
    'BEGIN': 0.0,          
    'END': 1.0,
    'DATA_FORMAT': 'NCDHW',
    'DO_PREPROCESS': False,
    'VIDEO_FPS': 30.0,
    'SIGNAL_FS': 500.0,
    'STATS_CSV': {'AVAILABLE': True, 'PATH': 'G:/preprocessed_data/preprocessed_cache_72x_150f_d-raw_l-raw_participant_stats.csv'},
    'PREPROCESS': {'DATA_TYPE': ['Raw'], 'LABEL_TYPE': 'Standardized'},
    'TEST': {'DATA': {'FS': 30.0}, 'METRICS': ['MAE', 'RMSE', 'MAPE', 'Pearson', 'SNR']},
    'INFERENCE': {'EVALUATION_METHOD': 'FFT', 'SAVE_PATH': OUTPUT_DIR, 'EVALUATION_WINDOW': {'USE_SMALLER_WINDOW': False}}
}

config_data = ConfigMock(loader_config_dict)

print("Loading Dataset...")
full_dataset = InfantDataLoader(
    dataset_name="Infant_Dataset",
    raw_data_path="G:/image/relax",
    label_data_path="G:/pulse_data/relax",
    config_data=config_data,
    device=DEVICE
)

groups = full_dataset.groups
dataset_indices = np.arange(len(full_dataset))

# Initialize KFold
n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

hyperparams = {
    'model_name': MODEL_NAME,
    'epochs': EPOCHS,
    'lr': LR,
    'w_mse': WEIGHT_MSE,
    'w_np': WEIGHT_NEG_PEARSON,
    'device': DEVICE
}

all_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(dataset_indices, groups=groups)):
    print(f"\n==========================================")
    print(f"          STARTING FOLD {fold_idx + 1}/{n_splits}         ")
    print(f"==========================================")
    
    train_sampler = SubsetRandomSampler(train_idx)
    val_sampler = SubsetRandomSampler(val_idx)

    train_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS, prefetch_factor=PREFETCH_FACTOR)
    val_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, sampler=val_sampler, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS, prefetch_factor=PREFETCH_FACTOR)

    # Initialize Model
    model = iBVPNet(frames=FRAMES, in_channels=IN_CHANNELS).to(DEVICE)
    
    # Run Training for this fold
    best_fold_metrics = train_and_validate(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        fold_idx=fold_idx + 1, 
        hyperparams=hyperparams, 
        config=config_data
    )
    
    all_fold_metrics.append(best_fold_metrics)
    print(f"Metrics (Lowest Val Loss) for Fold {fold_idx + 1}: {best_fold_metrics}")

print("\n==========================================")
print("          FINAL K-FOLD RESULTS            ")
print("==========================================")

# Extract just the values from the dictionary array
# Structure of best_fold_metrics: {"MAE": {"value": 0.5, "se": 0.01}, "RMSE": ...}
metric_keys = all_fold_metrics[0].keys()
aggregated = {key: [] for key in metric_keys}

for fold_res in all_fold_metrics:
    for key in metric_keys:
        aggregated[key].append(fold_res[key]["value"])
        
for key in metric_keys:
    mean_val = np.mean(aggregated[key])
    std_val = np.std(aggregated[key])
    print(f"{key}: {mean_val:.4f} ± {std_val:.4f}")

Loading Dataset...
Cached Data Path G:/preprocessed_data/preprocessed_cache_72x_150f_d-raw_l-raw

File List Path G:/preprocessed_data/preprocessed_cache_72x_150f_d-raw_l-raw.csv
 Infant_Dataset Preprocessed Dataset Length: 1896


          STARTING FOLD 1/5         

--- Fold 1 | Epoch 1/1 ---


Validation: 100%|███████████████████████████████| 49/49 [01:48<00:00,  2.21s/it]


Train Loss: 1.4945 | Val Loss: 1.4806
Calculating metrics!


100%|████████████████████████████████████████████| 3/3 [15:13<00:00, 304.52s/it]


AttributeError: 'ConfigMock' object has no attribute 'TOOLBOX_MODE'